# MICAI 2026 · MLOps Workshop
## Notebook 2 of 3 · Model Training and Experiment Tracking
**Chihuahua, Mexico · November 2, 2026**

| | |
|---|---|
| **Workshop** | MICAI 2026 · MLOps Workshop: from raw data to a monitored model |
| **Dataset** | *Wine Quality* (white wine). P. Cortez, A. Cerdeira, F. Almeida, T. Matos and J. Reis (2009). *Modeling wine preferences by data mining from physicochemical properties.* Decision Support Systems 47(4), 547–553. Obtained from the UCI Machine Learning Repository. The file `modified.csv` is a deliberately corrupted copy prepared for this workshop. |
| **Tools** | Python, pandas, scikit-learn, SciPy, MLflow, Databricks and Google Colab |

**Declaration on the use of AI.** These materials were developed with the assistance of generative AI
(Anthropic's Claude, model Claude Opus 5.5), used to draft and review code, explanations and documentation.
The workshop instructors reviewed and validated all content and are responsible for it.

    ## What you will learn

    Training a model is easy; knowing **which** model, trained with **which** settings on **which** data, gave
    **which** result is the hard part once dozens of experiments pile up. In this notebook you will:

    1. Load the **official dataset versions** from MLflow and verify they are the exact published files.
    2. Define the prediction task and choose **metrics that work with imbalanced data**.
    3. Build training sets that **cannot leak** information from the evaluation set.
    4. Train **random forest** models and **track** every run in MLflow: parameters, metrics, datasets, plots and the
       model itself.
    5. Compare the whole class on a shared **leaderboard** and answer: *does the cleaning strategy change the model?*

    **The workshop at a glance**

    | # | Notebook | Question it answers | MLflow experiment |
|---|---|---|---|
| 1 | Data quality, EDA and versioning | Can we trust the data? | `01_data_cleaning` |
| 2 | **➡ Training and experiment tracking** | Which model, trained on which data version, works best? | `02_training` |
| 3 | Monitoring and data drift | Is the model still valid once the data changes? | `03_drift_monitoring` |

    **How to use this notebook**
    - Run the cells from top to bottom: click a cell and press **Shift + Enter** (or use the ▶ button).
    - Every code cell is preceded by a text cell that explains what it does and why.
    - Sections marked **Deep dive (optional)** at the end are for later; they are not needed for the live session.

## Step 0 · Setup (about 1 minute)

Run the next cell. It shows two fields, or asks for them if you leave them empty:

1. **CONFIG_LINK**: paste the config link shown on the slide (or scanned from the QR code).
2. **NICKNAME**: any nickname for the class leaderboard. Please don't use your real name.

The cell then, automatically:
- downloads the workshop code and data from GitHub into this Colab machine,
- installs MLflow,
- downloads the workshop config, which contains the address of the shared MLflow server on Databricks and a temporary
  access key (the key is never printed),
- connects this notebook to the shared MLflow experiment `02_training`.

If anything fails (a mistyped link, the network, an expired key), the cell says so in one line and switches to a
**local** MLflow store inside Colab. The notebook still runs end to end; only the shared leaderboard is missing.

> Each notebook runs on its own Colab machine, so this setup is repeated at the top of every notebook,
> with the same link and nickname.

In [ ]:
#@title Step 0 · Workshop setup (click ▶ to run)
CONFIG_LINK = ""  #@param {type:"string"}
NICKNAME = ""  #@param {type:"string"}

import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/ITESM-MNA/MICAI_MLOps_2026.git"
try:
    import google.colab  # noqa: F401  (only available inside Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = Path("/content/MICAI_MLOps_2026")
    if not REPO_DIR.exists():
        print("Downloading the workshop files...")
        subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL, str(REPO_DIR)], check=True)
    print("Installing MLflow (about 20 seconds)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements-colab.txt")], check=True)
else:  # running outside Colab, from the notebooks/ folder of the repository
    here = Path.cwd()
    REPO_DIR = Path(os.environ.get("MICAI_REPO_DIR", here.parent if (here.parent / "src").exists() else here))

sys.path.insert(0, str(REPO_DIR / "src"))
from micai_mlops.setup import setup_workshop
ws = setup_workshop(CONFIG_LINK, NICKNAME, notebook="02", repo_dir=REPO_DIR)

## 1. Load the official datasets

In notebook 1 everyone produced their own cleaned versions. To make results comparable, all models in this notebook
are trained and evaluated on **one official copy**, published by the instructor in the MLflow experiment
`00_official_datasets`:

| Version | Content |
|---|---|
| `v1_clean_drop` | Strategy A: rows with any invalid feature dropped (real measurements only) |
| `v2_clean_impute` | Strategy B: invalid features replaced with the column median |
| `holdout` | Fixed evaluation set: 20% of v1, drawn once, never used for training |

The next cell downloads the files of that exact MLflow run and checks each **fingerprint** against the values
stored in the repository. If MLflow cannot be reached, it rebuilds the same files from the raw data with the same
code; the fingerprints prove they are identical.

In [ ]:
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import skops
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, average_precision_score, f1_score,
                             precision_score, recall_score, roc_auc_score)

from micai_mlops.contract import FEATURES, GOOD_THRESHOLD, TARGET
from micai_mlops.official import load_official
from micai_mlops.splits import training_set, xy

sns.set_theme(style="whitegrid")
official, fingerprint_check = load_official(ws)
fingerprint_check

## 2. The prediction task and how to measure it

**Task:** predict whether a wine is **good** (`quality ≥ 7`) from its 11 lab measurements. It is a binary
classification; predicting the exact score 3–9 would leave very few examples for the extreme classes.

Only about **1 wine in 5 is good**. With imbalanced data, **accuracy is misleading**: a "model" that always answers
*not good* is right about 80% of the time while being useless. We therefore track:

| Metric | Question it answers |
|---|---|
| **ROC-AUC** | How well does the model *rank* good wines above the rest? (0.5 = random, 1 = perfect). Our main metric. |
| **PR-AUC** | Same idea, focused on the rare *good* class (baseline = share of good wines) |
| **Precision** | Of the wines flagged as good, how many really are? |
| **Recall** | Of the truly good wines, how many did we find? |
| **F1** | Balance between precision and recall |

In [ ]:
holdout = official["holdout"]
X_holdout, y_holdout = xy(holdout, FEATURES)

summary = pd.DataFrame({name: {"rows": len(df), "% good": round(100 * (df[TARGET] >= GOOD_THRESHOLD).mean(), 1)}
                        for name, df in official.items()}).T
display(summary)

always_not_good = np.zeros(len(y_holdout), dtype=int)
print(f"Accuracy of always answering 'not good': {accuracy_score(y_holdout, always_not_good):.1%}  "
      f"(recall {recall_score(y_holdout, always_not_good):.0%}) -> useless, yet 'accurate'")

## 3. Training sets without leakage

**Data leakage** happens when information from the evaluation data reaches training: scores look great and collapse
in production. Two sources of leakage are removed from each training set:

1. **The same record:** every holdout `row_id` is removed from v1 and v2.
2. **Near-duplicates:** rows that match a holdout row in all but one column. They exist because the raw file contained
   injected duplicates; in v2 a duplicate whose one corrupted cell was *imputed* survives deduplication and would be
   an almost exact copy of a holdout wine.

Both versions are then evaluated on the **same holdout of real measurements**, so their scores are comparable.

In [ ]:
train_sets, leakage_report = {}, {}
for version in ["v1_clean_drop", "v2_clean_impute"]:
    train_sets[version], leakage_report[version] = training_set(official[version], holdout)
pd.DataFrame(leakage_report).T

## 4. Train a model and track it with MLflow

The function below trains a **random forest**: an ensemble of decision trees, each trained on a random sample of
rows and features, that vote together. `class_weight="balanced"` gives the rare *good* class more weight.

Everything needed to reproduce and judge the run is logged inside `mlflow.start_run()`:

| What | MLflow call | Where to see it in the UI |
|---|---|---|
| Hyperparameters and data version | `log_params` | Parameters |
| Holdout metrics | `log_metrics` | Metrics, and columns of the runs table |
| Training and evaluation datasets | `log_input` | Datasets column (with digest) |
| Confusion matrix and feature importance | `log_figure` | Artifacts → `evaluation.png` |
| The trained model, with its input example | `mlflow.sklearn.log_model` | Artifacts → `model` |
| Your nickname | `set_tags` | Tags, used by the leaderboard |

The model is saved in the **skops** format, a safer alternative to pickle. `skops_trusted_types` declares the one
scikit-learn internal type a random forest needs.

In [ ]:
def train_and_log(dataset_version, n_estimators=200, max_depth=None, min_samples_leaf=1, show=False):
    """Train a random forest on one data version, evaluate it on the holdout and log everything to MLflow."""
    train = train_sets[dataset_version]
    X_train, y_train = xy(train, FEATURES)
    params = {"n_estimators": n_estimators, "max_depth": max_depth, "min_samples_leaf": min_samples_leaf}

    model = RandomForestClassifier(**params, class_weight="balanced", random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

    proba = model.predict_proba(X_holdout)[:, 1]          # probability of "good"
    pred = (proba >= 0.5).astype(int)
    metrics = {"roc_auc": roc_auc_score(y_holdout, proba), "pr_auc": average_precision_score(y_holdout, proba),
               "f1": f1_score(y_holdout, pred), "precision": precision_score(y_holdout, pred, zero_division=0),
               "recall": recall_score(y_holdout, pred), "accuracy": accuracy_score(y_holdout, pred)}

    depth = "none" if max_depth is None else max_depth
    run_name = f"{ws.alias}-{dataset_version[:2]}-trees{n_estimators}-depth{depth}-leaf{min_samples_leaf}"
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tags({"alias": ws.alias, "role": "participant", "dataset_version": dataset_version})
        mlflow.log_params({**params, "dataset_version": dataset_version, "class_weight": "balanced",
                           "threshold": 0.5})
        mlflow.log_metrics(metrics)
        mlflow.log_input(mlflow.data.from_pandas(train, name=f"{dataset_version}_train", targets=TARGET),
                         context="training")
        mlflow.log_input(mlflow.data.from_pandas(holdout, name="holdout", targets=TARGET), context="evaluation")

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        ConfusionMatrixDisplay.from_predictions(y_holdout, pred, display_labels=["not good", "good"],
                                                ax=axes[0], colorbar=False)
        axes[0].set_title(f"Holdout confusion matrix (F1 = {metrics['f1']:.2f})")
        pd.Series(model.feature_importances_, FEATURES).sort_values().plot.barh(ax=axes[1], color="#4c72b0")
        axes[1].set_title("Feature importance")
        plt.tight_layout()
        mlflow.log_figure(fig, "evaluation.png")
        plt.show() if show else plt.close(fig)

        mlflow.sklearn.log_model(model, name="model", input_example=X_train.head(3),
                                 skops_trusted_types=["sklearn.tree._tree.Tree"],
                                 pip_requirements=[f"scikit-learn=={sklearn.__version__}",
                                                   f"skops=={skops.__version__}"])
    return {"run": run_name, "dataset_version": dataset_version, **params, "max_depth": depth,
            **{k: round(v, 3) for k, v in metrics.items()}, "run_id": run.info.run_id}

### Your experiment
Choose a data version and hyperparameters with the controls, then run the cell. There is no single right answer:
if everyone picks something different, the class leaderboard compares many more options than one person could.

- **N_ESTIMATORS**: number of trees. More trees give more stable votes, at the cost of time.
- **MAX_DEPTH**: maximum depth of each tree (`0` = no limit). Deeper trees learn finer patterns and can overfit.
- **MIN_SAMPLES_LEAF**: minimum number of wines in a leaf. Larger values make smoother, more cautious trees.

In [ ]:
#@title Your experiment
DATASET_VERSION = "v2_clean_impute"  #@param ["v1_clean_drop", "v2_clean_impute"]
N_ESTIMATORS = 200  #@param {type:"slider", min:50, max:400, step:50}
MAX_DEPTH = 0  #@param {type:"slider", min:0, max:20, step:2}
MIN_SAMPLES_LEAF = 3  #@param {type:"slider", min:1, max:20, step:1}

my_run = train_and_log(DATASET_VERSION, N_ESTIMATORS, MAX_DEPTH or None, MIN_SAMPLES_LEAF, show=True)
pd.Series(my_run).to_frame("your run")

## 5. A small grid: does the cleaning strategy matter?

To compare the two data versions fairly, we train the **same** two model configurations on each of them. Data
versions are treated just like hyperparameters: one more setting of the experiment, recorded in every run.

In [ ]:
grid = [{"max_depth": None, "min_samples_leaf": 5}, {"max_depth": 8, "min_samples_leaf": 1}]
results = [my_run]
for version in ["v1_clean_drop", "v2_clean_impute"]:
    for config in grid:
        results.append(train_and_log(version, n_estimators=200, **config))
        print(f"✅ {results[-1]['run']}: ROC-AUC {results[-1]['roc_auc']}, F1 {results[-1]['f1']}")

my_results = pd.DataFrame(results).drop(columns="run_id").sort_values("roc_auc", ascending=False)
my_results

## 6. Compare runs: the class leaderboard

Every run of every participant is in the same MLflow experiment. The next cell asks MLflow for all finished runs,
sorted by ROC-AUC. The run tagged `reference` is the instructor's **production model**, monitored in notebook 3.

In [ ]:
runs = mlflow.search_runs(experiment_names=[ws.experiment("training")],
                          filter_string="attributes.status = 'FINISHED'",
                          order_by=["metrics.roc_auc DESC"], max_results=2000)
columns = {"tags.alias": "nickname", "params.dataset_version": "data version", "params.n_estimators": "trees",
           "params.max_depth": "max depth", "params.min_samples_leaf": "min leaf",
           "metrics.roc_auc": "ROC-AUC", "metrics.pr_auc": "PR-AUC", "metrics.f1": "F1", "metrics.recall": "recall"}
board = runs.reindex(columns=list(columns)).rename(columns=columns)
print(f"{len(board)} runs from {board['nickname'].nunique()} participants")
display(board.head(10).round(3).reset_index(drop=True))

print("Average result by data version (whole class):")
board.groupby("data version")[["ROC-AUC", "PR-AUC", "F1"]].agg(["mean", "max", "count"]).round(3)

**How to read it.** Compare the averages per data version. With the same model settings, v2 (imputation) has more
training rows and usually ranks wines slightly better (ROC-AUC), while v1 keeps only real measurements. The
difference is small, which is itself a finding: the cleaning strategy matters less than one might expect here, and
without tracking we could not tell. Notice also that settings with a similar ROC-AUC can have very different F1 and
recall: ROC-AUC measures ranking, F1 depends on the 0.5 decision threshold.

### Explore the runs in the MLflow UI

In [ ]:
if ws.online:
    print("Open the experiment:", ws.experiment_url("training"))
else:
    print("You are working offline: the MLflow UI is not available, but the tables above come from your local MLflow store.")

In the MLflow UI:
1. Sort the runs table by **roc_auc** (click the column header).
2. Tick two or three runs and click **Compare** to see parameters and metrics side by side.
3. Open a run: **Datasets** shows the data version and its digest; **Artifacts** holds `evaluation.png` and the
   `model` folder with everything needed to load it again.

## Summary

1. **Fix the data before you compare models:** one official, fingerprinted copy and one fixed holdout.
2. **Prevent leakage explicitly**, including near-duplicates.
3. **Pick metrics that fit the problem:** accuracy hides failure on imbalanced data.
4. **Track everything:** parameters, data version, metrics, plots and the model, so any result can be reproduced and
   any model can be reloaded later.

**Next: notebook 3**, where the production model meets data that has changed.

---
## Deep dive (optional): load a model back from MLflow
A logged model can be reloaded anywhere from its run, without the training code. This is how a deployment job or the
monitoring notebook gets the exact model that was evaluated.

In [ ]:
best = max(results, key=lambda r: r["roc_auc"])
reloaded = mlflow.sklearn.load_model(f"runs:/{best['run_id']}/model")
proba = reloaded.predict_proba(X_holdout)[:, 1]
print(f"Reloaded '{best['run']}': ROC-AUC on the holdout = {roc_auc_score(y_holdout, proba):.3f} "
      f"(logged value: {best['roc_auc']})")